# V4 Feature Engineering
#### Top-K 성능 향상을 위한 rank/percentile/flag 중심의 feature engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
import seaborn as sns

In [2]:
DATA_DIR = Path('../Data')
train = pd.read_csv(DATA_DIR / 'train_V3.csv')
test  = pd.read_csv(DATA_DIR / 'test_V3.csv')

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Fraud rate: {train['fraud'].mean():.2%}")

Train: (18000, 117)  |  Test: (12000, 116)
Fraud rate: 15.82%


In [3]:
# Train / Test 합치기 (타겟 컬럼 제외)
train['_is_train'] = 1
test['_is_train']  = 0

# test에는 fraud 컬럼 없음 → 임시 NaN 추가
if 'fraud' not in test.columns:
    test['fraud'] = np.nan

df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined shape: {df.shape}")

Combined shape: (30000, 118)


In [4]:
# ====================
# 1. 안전한 보조 함수
# ====================
# 비율 피처 생성 시 나눗셈 처리 함수
def safe_divide(a, b):
    return a / (b.replace(0, np.nan) + 1e-6)

# 상위 q분위 이상인지 여부 -> 0/1 flag
def make_top_flag(train_df, test_df, col, q, feat_name):
    thr = train_df[col].quantile(q)
    train_df[feat_name] = (train_df[col] >= thr).astype(int)
    test_df[feat_name] = (test_df[col] >= thr).astype(int)
    return thr

# column 값 z-score로 반환
def make_zscore(train_df, test_df, col, feat_name):
    mean_ = train_df[col].mean()
    std_ = train_df[col].std()
    if std_ == 0 or pd.isna(std_):
        train_df[feat_name] = 0.0
        test_df[feat_name] = 0.0
    else:
        train_df[feat_name] = (train_df[col] - mean_) / std_
        test_df[feat_name] = (test_df[col] - mean_) / std_

# column 값을 train 기준 percentile로 변환
def make_rank_pct(train_df, test_df, col, feat_name):
    sorted_train = np.sort(train_df[col].dropna().values)

    def pct_map(x):
        if pd.isna(x):
            return np.nan
        return np.searchsorted(sorted_train, x, side='right') / len(sorted_train)

    train_df[feat_name] = train_df[col].apply(pct_map)
    test_df[feat_name] = test_df[col].apply(pct_map)

# 연속형 변수 값 bin으로 구분
def make_bucket_from_train(train_df, test_df, col, q, feat_name):
    try:
        _, bins = pd.qcut(train_df[col], q=q, retbins=True, duplicates="drop")
        bins[0] = -np.inf
        bins[-1] = np.inf

        train_df[feat_name] = pd.cut(train_df[col], bins=bins, labels=False)
        test_df[feat_name] = pd.cut(test_df[col], bins=bins, labels=False)

        train_df[feat_name] = train_df[feat_name].astype("float")
        test_df[feat_name] = test_df[feat_name].astype("float")
    except:
        train_df[feat_name] = np.nan
        test_df[feat_name] = np.nan

In [5]:
# ==========================
# 2. Rank / Percentile 피처
# ==========================
rank_cols = [
    "payout_to_income_ratio",
    "claim_est_payout",
    "vehicle_price",
    "past_num_of_claims"
]

for col in rank_cols:
    if col in train.columns and col in test.columns:
        feat_name = f"{col}_rank_pct"
        make_rank_pct(train, test, col, feat_name)

In [6]:
# ====================
# 3. Top-X% Flag 피처
# ====================
top_flag_cols = [
    "payout_to_income_ratio",
    "claim_est_payout",
    "vehicle_price",
    "past_num_of_claims"
]

for col in top_flag_cols:
    if col in train.columns and col in test.columns:
        make_top_flag(train, test, col, 0.90, f"top10_{col}")
        make_top_flag(train, test, col, 0.95, f"top5_{col}")
        make_top_flag(train, test, col, 0.99, f"top1_{col}")

In [7]:
# ============================
# 4. Z-score / deviation 피처
# ============================
z_cols = [
    "payout_to_income_ratio",
    "claim_est_payout",
    "vehicle_price",
    "past_num_of_claims",
    "annual_income"
]

for col in z_cols:
    if col in train.columns and col in test.columns:
        feat_name = f"{col}_z"
        make_zscore(train, test, col, feat_name)

In [8]:
# ===============================
# 5. Multi-condition fraud combo
# ===============================
if "payout_to_income_ratio" in train.columns:
    thr_payout_income_90 = train["payout_to_income_ratio"].quantile(0.90)
else:
    thr_payout_income_90 = None

if "claim_est_payout" in train.columns:
    thr_claim_90 = train["claim_est_payout"].quantile(0.90)
else:
    thr_claim_90 = None

if "vehicle_price" in train.columns:
    thr_vehicle_90 = train["vehicle_price"].quantile(0.90)
else:
    thr_vehicle_90 = None

if "annual_income" in train.columns:
    thr_income_30 = train["annual_income"].quantile(0.30)
else:
    thr_income_30 = None

# combo 1: high payout + no witness + no police report
needed_1 = ["payout_to_income_ratio", "witness_absent", "policy_report_filed_ind"]
if all(col in train.columns for col in needed_1) and all(col in test.columns for col in needed_1):
    train["fraud_combo_1"] = (
        (train["payout_to_income_ratio"] >= thr_payout_income_90) &
        (train["witness_absent"] == 1) &
        (train["policy_report_filed_ind"] == 0)
    ).astype(int)

    test["fraud_combo_1"] = (
        (test["payout_to_income_ratio"] >= thr_payout_income_90) &
        (test["witness_absent"] == 1) &
        (test["policy_report_filed_ind"] == 0)
    ).astype(int)

# combo 2: repeat claims + high liability + high payout
needed_2 = ["past_num_of_claims", "liab_prct", "payout_to_income_ratio"]
if all(col in train.columns for col in needed_2) and all(col in test.columns for col in needed_2):
    train["fraud_combo_2"] = (
        (train["past_num_of_claims"] >= 3) &
        (train["liab_prct"] >= 75) &
        (train["payout_to_income_ratio"] >= thr_payout_income_90)
    ).astype(int)

    test["fraud_combo_2"] = (
        (test["past_num_of_claims"] >= 3) &
        (test["liab_prct"] >= 75) &
        (test["payout_to_income_ratio"] >= thr_payout_income_90)
    ).astype(int)

# combo 3: low income + expensive vehicle + large claim
needed_3 = ["annual_income", "vehicle_price", "claim_est_payout"]
if all(col in train.columns for col in needed_3) and all(col in test.columns for col in needed_3):
    train["fraud_combo_3"] = (
        (train["annual_income"] <= thr_income_30) &
        (train["vehicle_price"] >= thr_vehicle_90) &
        (train["claim_est_payout"] >= thr_claim_90)
    ).astype(int)

    test["fraud_combo_3"] = (
        (test["annual_income"] <= thr_income_30) &
        (test["vehicle_price"] >= thr_vehicle_90) &
        (test["claim_est_payout"] >= thr_claim_90)
    ).astype(int)

# combo 4: repeat claims + no witness + no report
needed_4 = ["past_num_of_claims", "witness_absent", "policy_report_filed_ind"]
if all(col in train.columns for col in needed_4) and all(col in test.columns for col in needed_4):
    train["fraud_combo_4"] = (
        (train["past_num_of_claims"] >= 2) &
        (train["witness_absent"] == 1) &
        (train["policy_report_filed_ind"] == 0)
    ).astype(int)

    test["fraud_combo_4"] = (
        (test["past_num_of_claims"] >= 2) &
        (test["witness_absent"] == 1) &
        (test["policy_report_filed_ind"] == 0)
    ).astype(int)

In [9]:
# =============================
# 6. Ratio 간 비율 (2차 ratio)
# =============================
if all(col in train.columns for col in ["payout_to_income_ratio", "vehicle_price"]):
    train["payout_vs_price_ratio"] = safe_divide(train["payout_to_income_ratio"], train["vehicle_price"] + 1)
    test["payout_vs_price_ratio"] = safe_divide(test["payout_to_income_ratio"], test["vehicle_price"] + 1)

if all(col in train.columns for col in ["claim_est_payout", "annual_income", "vehicle_price"]):
    train["claim_over_income_plus_price"] = safe_divide(
        train["claim_est_payout"],
        train["annual_income"] + train["vehicle_price"] + 1
    )
    test["claim_over_income_plus_price"] = safe_divide(
        test["claim_est_payout"],
        test["annual_income"] + test["vehicle_price"] + 1
    )

if all(col in train.columns for col in ["claim_est_payout", "policy_duration"]):
    train["claim_per_policy_duration"] = safe_divide(train["claim_est_payout"], train["policy_duration"] + 1)
    test["claim_per_policy_duration"] = safe_divide(test["claim_est_payout"], test["policy_duration"] + 1)

In [10]:
# ========================
# 7. Bucket 피처 (구간화)
# ========================
bucket_cols = [
    "annual_income",
    "claim_est_payout",
    "vehicle_price",
    "past_num_of_claims"
]

for col in bucket_cols:
    if col in train.columns and col in test.columns:
        feat_name = f"{col}_bucket_q5"
        make_bucket_from_train(train, test, col, q=5, feat_name=feat_name)

# low income bucket x high payout
needed_bucket_combo = ["annual_income_bucket_q5", "claim_est_payout"]
if all(col in train.columns for col in needed_bucket_combo) and all(col in test.columns for col in needed_bucket_combo):
    claim_thr_90 = train["claim_est_payout"].quantile(0.90)

    train["low_income_bucket_high_claim"] = (
        (train["annual_income_bucket_q5"] == train["annual_income_bucket_q5"].min()) &
        (train["claim_est_payout"] >= claim_thr_90)
    ).astype(int)

    test["low_income_bucket_high_claim"] = (
        (test["annual_income_bucket_q5"] == train["annual_income_bucket_q5"].min()) &
        (test["claim_est_payout"] >= claim_thr_90)
    ).astype(int)

In [11]:
# ===================
# 8. Count 기반 조합
# ===================
if all(col in train.columns for col in ["past_num_of_claims", "claim_est_payout"]):
    thr_claim_90 = train["claim_est_payout"].quantile(0.90)

    train["repeat_high_payout"] = (
        (train["past_num_of_claims"] >= 3) &
        (train["claim_est_payout"] >= thr_claim_90)
    ).astype(int)

    test["repeat_high_payout"] = (
        (test["past_num_of_claims"] >= 3) &
        (test["claim_est_payout"] >= thr_claim_90)
    ).astype(int)

In [12]:
# ====================
# 9. Interaction 피처
# ====================
if all(col in train.columns for col in ["annual_income", "vehicle_price"]):
    train["income_x_vehicle"] = train["annual_income"] * train["vehicle_price"]
    test["income_x_vehicle"] = test["annual_income"] * test["vehicle_price"]

if all(col in train.columns for col in ["payout_to_income_ratio_rank_pct", "past_num_of_claims"]):
    train["risk_score_interaction"] = train["payout_to_income_ratio_rank_pct"] * train["past_num_of_claims"]
    test["risk_score_interaction"] = test["payout_to_income_ratio_rank_pct"] * test["past_num_of_claims"]

if all(col in train.columns for col in ["claim_est_payout_rank_pct", "liab_prct"]):
    train["claim_liability_interaction"] = train["claim_est_payout_rank_pct"] * train["liab_prct"]
    test["claim_liability_interaction"] = test["claim_est_payout_rank_pct"] * test["liab_prct"]

In [13]:
# Train / Test 재분리
train_fe = df[df['_is_train'] == 1].drop(columns=['_is_train']).copy()
test_fe  = df[df['_is_train'] == 0].drop(columns=['_is_train', 'fraud']).copy()

print(f"Train 피처 엔지니어링 완료: {train_fe.shape}")
print(f"Test  피처 엔지니어링 완료: {test_fe.shape}")

Train 피처 엔지니어링 완료: (18000, 117)
Test  피처 엔지니어링 완료: (12000, 116)


In [14]:
# 최종 데이터셋 저장
X_train = train_fe.drop(columns=['fraud']).copy()
y_train = train_fe['fraud']
X_test  = test_fe

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}  (fraud={y_train.mean():.2%})")
print(f"X_test:  {X_test.shape}")

# CSV 저장 (V4 데이터셋)
X_train.assign(fraud=y_train.values).to_csv(DATA_DIR / 'train_V4.csv', index=False)
X_test.to_csv(DATA_DIR / 'test_V4.csv', index=False)

X_train: (18000, 116)
y_train: (18000,)  (fraud=15.82%)
X_test:  (12000, 116)
